# Music Annotation Project V8.8F
## Notebook 57: Release, Control and Notebook 09 Disposition

**Runtime:** CPU. Select **Runtime → Run all**.

Notebook 57 is the release-decision and control notebook for the V8.8 M5_REMOVE programme. It consumes the completed Notebook 52–56 evidence chain, verifies exact policy and implementation identity, validates the supervised-pilot approval record, creates a release candidate manifest and writes an explicit disposition for Notebook 09.

### Fixed governance contract
- No retraining, recalibration, threshold movement or SAFE_PROTECT activation.
- Notebook 57 does not manufacture a release pass from missing evidence.
- The correction-engine decision is distinct from annotation-mask authority.
- Notebook 09 is never deleted. It remains a versioned rollback and evidence path.
- A supervised-pilot PASS does not silently authorise automatic annotated-only mask authority.
- The initial release decision defaults to **V8.8 primary correction engine with authoritative masks supplied**, while Notebook 09 remains fallback/exception review, only if every mandatory gate passes.
- Automatic annotated-only output remains disabled unless a separately validated mask-authority release is explicitly supplied.
- Failures are written before the notebook stops.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy pyarrow opencv-python-headless

## 1. Configuration and deliberate release controls

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, zipfile, sys, platform, os
import numpy as np
import pandas as pd
import cv2

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N52_ROOT=ROOT/'m5_remove_policy_freeze_v8_8a'
N53_ROOT=ROOT/'m5_remove_operational_builder_v8_8b'
N54_ROOT=ROOT/'m5_remove_exact_equivalence_v8_8c'
N55_ROOT=ROOT/'m5_remove_notebook09_shadow_v8_8d'
N56_ROOT=ROOT/'m5_remove_supervised_pilot_v8_8e'
OUTPUT_ROOT=ROOT/'m5_remove_release_control_v8_8f'
RELEASE_REGISTRY_ROOT=ROOT/'release_registry'/'m5_remove_v8_8'
ACTIVE_CONFIG_ROOT=ROOT/'active_release'

NOTEBOOK52_RUN_OVERRIDE=None
NOTEBOOK53_RUN_OVERRIDE=None
NOTEBOOK54_RUN_OVERRIDE=None
NOTEBOOK55_RUN_OVERRIDE=None
NOTEBOOK56_RUN_OVERRIDE=None

# Deliberate human release controls. Keep False for evidence-only dry run.
AUTHORISE_RELEASE=False
APPROVER_NAME=''
APPROVAL_REFERENCE=''
RELEASE_NOTES=''

# Allowed: PRIMARY_CORRECTION_ENGINE, SHADOW_ONLY, RETAIN_NOTEBOOK09_PRIMARY
REQUESTED_DISPOSITION='PRIMARY_CORRECTION_ENGINE'
MASK_AUTHORITY_MODE='AUTHORITATIVE_REVIEWED_MASK_REQUIRED'
AUTOMATIC_ANNOTATED_ONLY_ENABLED=False

# Optional scope limits. Empty means all pages covered by the passed supervised pilot.
RELEASE_PAGE_ALLOWLIST=[]
RELEASE_SOURCE_GROUP_ALLOWLIST=[]

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook57_v8_8f_release_control'
OUT=OUTPUT_ROOT/RUN
T=OUT/'tables'; A=OUT/'audit'; R=OUT/'release_candidate'; RB=OUT/'rollback'; U=OUT/'upload'
for p in [OUT,T,A,R,RB,U]: p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)
print('Authorise release:',AUTHORISE_RELEASE)
print('Requested disposition:',REQUESTED_DISPOSITION)


## 2. Helpers and fail-closed run resolution

In [ ]:
def require(condition,message):
    if not bool(condition): raise RuntimeError(message)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()

def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def read_json(path): return json.loads(Path(path).read_text())

def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp'); tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str)); tmp.replace(path)

def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def choose(override,root,required,label):
    p=Path(override) if override else latest_complete(root,required)
    require(p is not None and p.exists(),f'No complete {label} run found below {root}.')
    return p

def copy_checked(src,dst):
    require(Path(src).exists(),f'Missing release artefact: {src}')
    Path(dst).parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst)
    require(sha256_file(src)==sha256_file(dst),f'Copy hash mismatch: {src} -> {dst}')

def page_files(folder):
    supported={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
    return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in supported} if Path(folder).exists() else {}


## 3. Resolve the exact Notebook 52–56 evidence chain

In [ ]:
N56=choose(NOTEBOOK56_RUN_OVERRIDE,N56_ROOT,['notebook56_verdict.json','pilot_approval.json','tables/pilot_page_decisions.csv','final'],'Notebook 56')
v56=read_json(N56/'notebook56_verdict.json')
require(v56.get('verdict')=='PASS_SUPERVISED_PILOT_COMPLETE',f'Notebook 56 did not pass: {v56}')
require(v56.get('automatic_approval') is False,'Notebook 56 automatic-approval flag is not false.')

N55=Path(NOTEBOOK55_RUN_OVERRIDE) if NOTEBOOK55_RUN_OVERRIDE else Path(v56['notebook55_run'])
require(N55.exists(),'Notebook 55 source recorded by Notebook 56 is unavailable.')
v55=read_json(N55/'notebook55_verdict.json')
require(v55.get('verdict')=='PASS_SHADOW_EVIDENCE_READY_FOR_SUPERVISED_PILOT',f'Notebook 55 did not pass: {v55}')

N53=Path(NOTEBOOK53_RUN_OVERRIDE) if NOTEBOOK53_RUN_OVERRIDE else Path(v56['notebook53_run'])
require(N53.exists(),'Notebook 53 source recorded by Notebook 56 is unavailable.')
v53=read_json(N53/'notebook53_verdict.json'); id53=read_json(N53/'experiment_identity.json')
require(str(v53.get('verdict','')).startswith('PASS_'),f'Notebook 53 did not pass: {v53}')

N54=Path(NOTEBOOK54_RUN_OVERRIDE) if NOTEBOOK54_RUN_OVERRIDE else Path(v55['notebook54_run'])
require(N54.exists(),'Notebook 54 source recorded by Notebook 55 is unavailable.')
v54=read_json(N54/'notebook54_verdict.json')
require(str(v54.get('verdict','')).startswith('PASS_') or v54.get('verdict')=='PASS',f'Notebook 54 did not pass: {v54}')

N52=Path(NOTEBOOK52_RUN_OVERRIDE) if NOTEBOOK52_RUN_OVERRIDE else Path(id53['notebook52_run']) if id53.get('notebook52_run') else choose(None,N52_ROOT,['notebook52_verdict.json','policy_bundle/policy_manifest.json'],'Notebook 52')
require(N52.exists(),'Notebook 52 source unavailable.')
v52=read_json(N52/'notebook52_verdict.json'); policy52=read_json(N52/'policy_bundle'/'policy_manifest.json')
require(v52.get('verdict')=='PASS_M5_REMOVE_FREEZE_READY_FOR_OPERATIONAL_BUILDER',f'Notebook 52 did not pass: {v52}')

N09=Path(v55['notebook09_run']); require(N09.exists(),'Notebook 09 source recorded by Notebook 55 is unavailable.')
policy_id=id53['policy_id']; implementation_id=id53['implementation_id']
print('N52:',N52);print('N53:',N53);print('N54:',N54);print('N55:',N55);print('N56:',N56);print('N09:',N09)
print('Policy:',policy_id);print('Implementation:',implementation_id)


## 4. Identity and immutable-chain audit

In [ ]:
identity_checks={
 'n52_policy_matches_n53':policy52.get('policy_id')==policy_id,
 'n53_policy_matches_n55':v55.get('policy_id')==policy_id,
 'n53_policy_matches_n56':v56.get('policy_id')==policy_id,
 'n53_implementation_matches_n55':v55.get('implementation_id')==implementation_id,
 'n53_implementation_matches_n56':v56.get('implementation_id')==implementation_id,
 'safe_protect_disabled':policy52.get('safe_protect_runtime_enabled') is False,
 'no_retraining':policy52.get('retraining_performed') is False and id53.get('retrained') is False,
 'no_recalibration':policy52.get('recalibration_performed') is False and id53.get('recalibrated') is False,
 'no_threshold_change':policy52.get('threshold_changed') is False and id53.get('threshold_changed') is False,
 'n55_no_automatic_authority':v55.get('automatic_output_authority') is False,
 'n56_no_automatic_approval':v56.get('automatic_approval') is False,
}
pd.DataFrame([{'check':k,'passed':v} for k,v in identity_checks.items()]).to_csv(T/'identity_checks.csv',index=False)
require(all(identity_checks.values()),f'Identity/control failure: {identity_checks}')

chain_files={
 'notebook52_verdict':N52/'notebook52_verdict.json',
 'notebook52_policy_manifest':N52/'policy_bundle'/'policy_manifest.json',
 'notebook53_verdict':N53/'notebook53_verdict.json',
 'notebook53_identity':N53/'experiment_identity.json',
 'notebook54_verdict':N54/'notebook54_verdict.json',
 'notebook55_verdict':N55/'notebook55_verdict.json',
 'notebook56_verdict':N56/'notebook56_verdict.json',
 'pilot_approval':N56/'pilot_approval.json',
 'pilot_page_decisions':N56/'tables'/'pilot_page_decisions.csv',
}
chain_rows=[]
for role,p in chain_files.items():
    require(p.exists(),f'Missing chain file: {p}')
    chain_rows.append({'role':role,'path':str(p),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
chain=pd.DataFrame(chain_rows); chain.to_csv(T/'evidence_chain_inventory.csv',index=False)
chain_id=hashlib.sha256(chain.sort_values('role')[['role','sha256']].to_csv(index=False).encode()).hexdigest()
print('Evidence chain ID:',chain_id)

## 5. Pilot outcome, release scope and final-page verification

In [ ]:
decisions=pd.read_csv(N56/'tables'/'pilot_page_decisions.csv',dtype=str)
required={'page','decision','approved_source','final_path','final_sha256'}
require(required.issubset(decisions.columns),f'Pilot decision table missing: {sorted(required-set(decisions.columns))}')
require(not decisions.page.duplicated().any(),'Pilot decision pages are duplicated.')
require(set(decisions.approved_source).issubset({'V8.8','Notebook09'}),f'Unexpected approved source: {sorted(set(decisions.approved_source))}')

if RELEASE_PAGE_ALLOWLIST:
    missing=sorted(set(RELEASE_PAGE_ALLOWLIST)-set(decisions.page))
    require(not missing,f'Release allowlist contains pages absent from pilot: {missing}')
    scope=decisions[decisions.page.isin(RELEASE_PAGE_ALLOWLIST)].copy()
else:
    scope=decisions.copy()
require(len(scope)>0,'Release scope is empty.')

verify_rows=[]
for _,row in scope.iterrows():
    p=Path(row.final_path)
    if not p.exists(): p=N56/'final'/f'{row.page}.png'
    require(p.exists(),f'{row.page}: supervised-pilot final page is missing.')
    actual=sha256_file(p)
    require(actual==row.final_sha256,f'{row.page}: final-page hash no longer matches pilot decision.')
    expected=N53/'pages'/row.page/'corrected.png' if row.approved_source=='V8.8' else N09/'final'/f'{row.page}.png'
    if not expected.exists():
        hits=list((N09/'final').glob(row.page+'.*')) if row.approved_source=='Notebook09' else []
        require(len(hits)==1,f'{row.page}: cannot resolve expected approved source page.'); expected=hits[0]
    require(sha256_file(expected)==actual,f'{row.page}: final page does not match approved source {row.approved_source}.')
    verify_rows.append({'page':row.page,'decision':row.decision,'approved_source':row.approved_source,'final_path':str(p),'final_sha256':actual,'expected_source_path':str(expected),'exact_source_match':True})
verified=pd.DataFrame(verify_rows); verified.to_csv(T/'release_scope_page_verification.csv',index=False)

pilot_summary={
 'pages':len(scope),
 'v88_accepted':int((scope.approved_source=='V8.8').sum()),
 'notebook09_retained':int((scope.approved_source=='Notebook09').sum()),
 'mask_review_required':int((scope.decision=='mask_review_required').sum()),
 'all_final_hashes_verified':bool(verified.exact_source_match.all()),
}
atomic_json(A/'pilot_release_scope_summary.json',pilot_summary)
print(json.dumps(pilot_summary,indent=2))


## 6. Release decision gates and Notebook 09 disposition

In [ ]:
require(REQUESTED_DISPOSITION in ['PRIMARY_CORRECTION_ENGINE','SHADOW_ONLY','RETAIN_NOTEBOOK09_PRIMARY'],'Invalid REQUESTED_DISPOSITION')
require(MASK_AUTHORITY_MODE in ['AUTHORITATIVE_REVIEWED_MASK_REQUIRED','SEPARATELY_VALIDATED_MASK_AUTHORITY'],'Invalid MASK_AUTHORITY_MODE')
require(not AUTOMATIC_ANNOTATED_ONLY_ENABLED or MASK_AUTHORITY_MODE=='SEPARATELY_VALIDATED_MASK_AUTHORITY','Automatic annotated-only output requires a separately validated mask-authority release.')

# Release gating deliberately treats pilot completion as evidence, not a requirement that every page chose V8.8.
# Retained Notebook 09 pages become explicit exceptions in the release scope.
gates={
 'notebook52_pass':v52.get('verdict')=='PASS_M5_REMOVE_FREEZE_READY_FOR_OPERATIONAL_BUILDER',
 'notebook53_pass':str(v53.get('verdict','')).startswith('PASS_'),
 'notebook54_exact_equivalence_pass':str(v54.get('verdict','')).startswith('PASS_') or v54.get('verdict')=='PASS',
 'notebook55_shadow_pass':v55.get('verdict')=='PASS_SHADOW_EVIDENCE_READY_FOR_SUPERVISED_PILOT',
 'notebook56_supervised_pilot_pass':v56.get('verdict')=='PASS_SUPERVISED_PILOT_COMPLETE',
 'identity_chain_exact':all(identity_checks.values()),
 'release_scope_nonempty':len(scope)>0,
 'release_scope_final_hashes_exact':bool(verified.exact_source_match.all()),
 'safe_protect_disabled':policy52.get('safe_protect_runtime_enabled') is False,
 'outside_m3_shadow_disagreement_zero':v55.get('outside_m3_disagreement_pixels')==0,
 'human_approval_present':(N56/'pilot_approval.json').exists(),
 'mask_authority_not_overclaimed':MASK_AUTHORITY_MODE in ['AUTHORITATIVE_REVIEWED_MASK_REQUIRED','SEPARATELY_VALIDATED_MASK_AUTHORITY'],
 'automatic_annotated_only_guarded':not AUTOMATIC_ANNOTATED_ONLY_ENABLED or MASK_AUTHORITY_MODE=='SEPARATELY_VALIDATED_MASK_AUTHORITY',
}
mandatory_order=list(gates)
first_failed=next((k for k in mandatory_order if gates[k] is not True),None)

if first_failed is not None:
    release_status='BLOCKED'
    disposition='RETAIN_NOTEBOOK09_PRIMARY'
elif REQUESTED_DISPOSITION=='PRIMARY_CORRECTION_ENGINE':
    release_status='READY_FOR_AUTHORISED_RELEASE' if not AUTHORISE_RELEASE else 'RELEASED'
    disposition='V8_8_PRIMARY_CORRECTION_ENGINE_WITH_NOTEBOOK09_FALLBACK'
elif REQUESTED_DISPOSITION=='SHADOW_ONLY':
    release_status='SHADOW_ONLY'
    disposition='V8_8_SHADOW_NOTEBOOK09_PRIMARY'
else:
    release_status='NO_PROMOTION'
    disposition='RETAIN_NOTEBOOK09_PRIMARY'

notebook09_role={
 'primary_correction_engine':disposition=='RETAIN_NOTEBOOK09_PRIMARY' or disposition=='V8_8_SHADOW_NOTEBOOK09_PRIMARY',
 'shadow_comparator':True,
 'exception_reviewer':True,
 'rollback_path':True,
 'mask_authority_and_mask_edit_path':MASK_AUTHORITY_MODE=='AUTHORITATIVE_REVIEWED_MASK_REQUIRED',
 'deletion_authorised':False,
}
pd.DataFrame([{'gate':k,'passed':v} for k,v in gates.items()]).to_csv(T/'notebook57_gate_results.csv',index=False)
atomic_json(R/'notebook09_disposition.json',{'created_utc':datetime.now(timezone.utc).isoformat(),'disposition':disposition,'roles':notebook09_role,'notebook09_run':str(N09),'reason':'V8.8 correction-engine decision is separated from mask authority and retained rollback/evidence obligations.'})
print('Status:',release_status);print('Disposition:',disposition);print('First failed gate:',first_failed)


## 7. Build the immutable release candidate and rollback record

In [ ]:
scope_rows=[]
for _,row in scope.iterrows():
    page=row.page
    package_page=R/'approved_pages'/page
    package_page.mkdir(parents=True,exist_ok=False)
    final_src=Path(row.final_path)
    if not final_src.exists(): final_src=N56/'final'/f'{page}.png'
    copy_checked(final_src,package_page/'approved.png')
    # Preserve the V8.8 audit arrays on every release page, even when Notebook 09 was retained.
    for name in ['observed.png','m3.png','safe_remove.png','m5_remove.png','corrected.png','difference_from_observed.png']:
        src=N53/'pages'/page/name
        require(src.exists(),f'{page}: missing Notebook 53 audit page {name}')
        copy_checked(src,package_page/name)
    scope_rows.append({'page':page,'approved_source':row.approved_source,'decision':row.decision,'approved_sha256':sha256_file(package_page/'approved.png'),'policy_id':policy_id,'implementation_id':implementation_id})
release_pages=pd.DataFrame(scope_rows); release_pages.to_csv(R/'release_page_manifest.csv',index=False)

# Copy compact evidence records, never overwrite the source runs.
for role,p in chain_files.items(): copy_checked(p,R/'evidence'/f'{role}{p.suffix}')
copy_checked(T/'identity_checks.csv',R/'evidence'/'identity_checks.csv')
copy_checked(T/'notebook57_gate_results.csv',R/'evidence'/'notebook57_gate_results.csv')
copy_checked(T/'release_scope_page_verification.csv',R/'evidence'/'release_scope_page_verification.csv')

rollback={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'trigger':'Any integrity, immutability, source-specific, mask-authority, monitoring or operational failure attributable to the V8.8 release.',
 'rollback_action':'Set primary correction engine to the recorded Notebook 09 run; retain the V8.8 run and incident evidence; do not delete or overwrite either lineage.',
 'notebook09_run':str(N09),
 'notebook09_final_directory':str(N09/'final'),
 'notebook09_manifest_sha256':sha256_file(N09/'pilot_manifest.json') if (N09/'pilot_manifest.json').exists() else None,
 'notebook53_run':str(N53),
 'notebook56_run':str(N56),
 'policy_id':policy_id,
 'implementation_id':implementation_id,
 'release_chain_id':chain_id,
}
atomic_json(RB/'rollback_plan.json',rollback)
copy_checked(RB/'rollback_plan.json',R/'rollback_plan.json')

release_manifest={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'version':'8.8F-notebook57-release-control',
 'release_status':release_status,
 'authorised':bool(AUTHORISE_RELEASE and release_status=='RELEASED'),
 'approver_name':APPROVER_NAME if AUTHORISE_RELEASE else None,
 'approval_reference':APPROVAL_REFERENCE if AUTHORISE_RELEASE else None,
 'release_notes':RELEASE_NOTES,
 'requested_disposition':REQUESTED_DISPOSITION,
 'effective_disposition':disposition,
 'policy_id':policy_id,
 'implementation_id':implementation_id,
 'evidence_chain_id':chain_id,
 'mask_authority_mode':MASK_AUTHORITY_MODE,
 'automatic_annotated_only_enabled':AUTOMATIC_ANNOTATED_ONLY_ENABLED,
 'notebook09_roles':notebook09_role,
 'pages':len(release_pages),
 'v88_pages':int((release_pages.approved_source=='V8.8').sum()),
 'notebook09_exception_pages':int((release_pages.approved_source=='Notebook09').sum()),
 'gates':gates,
 'first_failed_gate':first_failed,
 'source_runs':{'notebook52':str(N52),'notebook53':str(N53),'notebook54':str(N54),'notebook55':str(N55),'notebook56':str(N56),'notebook09':str(N09)},
 'release_page_manifest_sha256':sha256_file(R/'release_page_manifest.csv'),
 'rollback_plan_sha256':sha256_file(R/'rollback_plan.json'),
 'automatic_promotion':False,
 'safe_protect_enabled':False,
}
release_manifest['release_id']=hashlib.sha256(json.dumps({k:release_manifest[k] for k in ['policy_id','implementation_id','evidence_chain_id','mask_authority_mode','automatic_annotated_only_enabled','release_page_manifest_sha256','rollback_plan_sha256']},sort_keys=True).encode()).hexdigest()
atomic_json(R/'release_manifest.json',release_manifest)
print('Release candidate ID:',release_manifest['release_id'])

## 8. Optional activation and versioned registry write

In [ ]:
activation_record={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'release_id':release_manifest['release_id'],
 'release_status':release_status,
 'effective_disposition':disposition,
 'policy_id':policy_id,
 'implementation_id':implementation_id,
 'mask_authority_mode':MASK_AUTHORITY_MODE,
 'automatic_annotated_only_enabled':AUTOMATIC_ANNOTATED_ONLY_ENABLED,
 'release_manifest':str(R/'release_manifest.json'),
 'rollback_plan':str(R/'rollback_plan.json'),
 'notebook09_run':str(N09),
 'notebook53_run':str(N53),
 'notebook56_run':str(N56),
}

if AUTHORISE_RELEASE:
    require(first_failed is None,'Release cannot be authorised while a mandatory gate is false.')
    require(APPROVER_NAME.strip(),'APPROVER_NAME is required for an authorised release.')
    require(APPROVAL_REFERENCE.strip(),'APPROVAL_REFERENCE is required for an authorised release.')
    require(REQUESTED_DISPOSITION=='PRIMARY_CORRECTION_ENGINE','AUTHORISE_RELEASE is supported only for PRIMARY_CORRECTION_ENGINE in this notebook.')
    RELEASE_REGISTRY_ROOT.mkdir(parents=True,exist_ok=True)
    registry_dir=RELEASE_REGISTRY_ROOT/release_manifest['release_id']
    require(not registry_dir.exists(),f'Release ID already registered: {registry_dir}')
    shutil.copytree(R,registry_dir)
    ACTIVE_CONFIG_ROOT.mkdir(parents=True,exist_ok=True)
    active_path=ACTIVE_CONFIG_ROOT/'m5_remove_v8_8.json'
    # Preserve prior active pointer as a rollback audit item.
    if active_path.exists():
        previous=read_json(active_path); atomic_json(RB/'previous_active_release.json',previous)
    atomic_json(active_path,activation_record)
    require(read_json(active_path)['release_id']==release_manifest['release_id'],'Active release pointer verification failed.')
    activation_record['active_config_path']=str(active_path)
    activation_record['registry_path']=str(registry_dir)
    activation_record['activated']=True
else:
    activation_record['activated']=False
    activation_record['interpretation']='Dry run: release candidate and rollback pack created, but no active release pointer or registry entry was changed.'

atomic_json(OUT/'activation_record.json',activation_record)
print(json.dumps(activation_record,indent=2))


## 9. Monitoring and exception contract

In [ ]:
monitoring={
 'release_id':release_manifest['release_id'],
 'policy_id':policy_id,
 'implementation_id':implementation_id,
 'required_runtime_records':['input file identity','authoritative mask identity/status','M3 hash','SAFE_REMOVE hash','M5_REMOVE hash','corrected-page hash','reason/fallback status','software/configuration identity'],
 'mandatory_zero_tolerance':['outside-authority pixel changes','policy/schema/hash mismatch','partial page output','non-finite decision input','unauthorised SAFE_PROTECT use'],
 'exception_routes':['retain Notebook 09 output','mask review/edit and rescore','defer/no output','rollback release'],
 'review_dimensions':['page','source group','mask status','SAFE_REMOVE coverage','fallback reason','Notebook 09 disagreement'],
 'retraining_policy':'Event-driven challenger only. No calendar retraining and no threshold weakening to increase coverage.',
 'automatic_annotated_only_enabled':AUTOMATIC_ANNOTATED_ONLY_ENABLED,
 'mask_authority_mode':MASK_AUTHORITY_MODE,
 'notebook09_retained':True,
}
atomic_json(R/'monitoring_contract.json',monitoring)

exception_pages=release_pages[release_pages.approved_source=='Notebook09'].copy()
exception_pages.to_csv(R/'notebook09_exception_pages.csv',index=False)
print('Notebook 09 exception pages:',len(exception_pages))


## 10. Final verdict and curated upload package

In [ ]:
if first_failed is not None:
    verdict='FAIL_V8_8F_RELEASE_CONTROL'
elif AUTHORISE_RELEASE and activation_record.get('activated'):
    verdict='PASS_V8_8_RELEASED_PRIMARY_CORRECTION_ENGINE'
elif REQUESTED_DISPOSITION=='PRIMARY_CORRECTION_ENGINE':
    verdict='PASS_RELEASE_CANDIDATE_READY_FOR_AUTHORISED_ACTIVATION'
elif REQUESTED_DISPOSITION=='SHADOW_ONLY':
    verdict='PASS_V8_8_RETAINED_IN_SHADOW'
else:
    verdict='PASS_NOTEBOOK09_RETAINED_PRIMARY'

record={'created_utc':datetime.now(timezone.utc).isoformat(),'verdict':verdict,'first_failed_gate':first_failed,'gates':gates,'release_id':release_manifest['release_id'],'release_status':release_status,'authorised':bool(activation_record.get('activated')),'effective_disposition':disposition,'policy_id':policy_id,'implementation_id':implementation_id,'evidence_chain_id':chain_id,'pages':len(release_pages),'v88_pages':int((release_pages.approved_source=='V8.8').sum()),'notebook09_exception_pages':int((release_pages.approved_source=='Notebook09').sum()),'mask_authority_mode':MASK_AUTHORITY_MODE,'automatic_annotated_only_enabled':AUTOMATIC_ANNOTATED_ONLY_ENABLED,'notebook09_roles':notebook09_role,'meaning':'PASS creates a controlled release candidate, or activates it only when AUTHORISE_RELEASE plus named approval are supplied. Notebook 09 remains rollback, exception and evidence infrastructure.'}
atomic_json(OUT/'notebook57_verdict.json',record)

summary=['NOTEBOOK 57 / V8.8F RELEASE, CONTROL AND NOTEBOOK 09 DISPOSITION','',f'Verdict: {verdict}',f'Release ID: {release_manifest["release_id"]}',f'Release status: {release_status}',f'Authorised/activated: {activation_record.get("activated")}',f'Policy ID: {policy_id}',f'Implementation ID: {implementation_id}',f'Evidence chain ID: {chain_id}',f'Release pages: {len(release_pages)}',f'V8.8 approved pages: {int((release_pages.approved_source=="V8.8").sum())}',f'Notebook 09 exception pages: {int((release_pages.approved_source=="Notebook09").sum())}',f'Notebook 09 disposition: {disposition}',f'Mask authority: {MASK_AUTHORITY_MODE}',f'Automatic annotated-only enabled: {AUTOMATIC_ANNOTATED_ONLY_ENABLED}','','Notebook 09 is retained as shadow comparator, exception reviewer, rollback path and evidence. No automatic promotion occurred.']
(OUT/'executive_summary.txt').write_text(chr(10).join(summary))

upload_files=[OUT/'executive_summary.txt',OUT/'notebook57_verdict.json',OUT/'activation_record.json',R/'release_manifest.json',R/'notebook09_disposition.json',R/'rollback_plan.json',R/'monitoring_contract.json',R/'release_page_manifest.csv',R/'notebook09_exception_pages.csv',T/'notebook57_gate_results.csv',T/'identity_checks.csv',T/'evidence_chain_inventory.csv',T/'release_scope_page_verification.csv',A/'pilot_release_scope_summary.json']
rows=[]
for src in upload_files:
    require(src.exists(),f'Missing upload artefact: {src}')
    dst=U/src.name; copy_checked(src,dst); rows.append({'file':dst.name,'bytes':dst.stat().st_size,'sha256':sha256_file(dst),'source':str(src.relative_to(OUT))})
pd.DataFrame(rows).to_csv(U/'upload_manifest.csv',index=False)
(U/'README.txt').write_text('Notebook 57 V8.8F release-candidate, activation, Notebook 09 disposition, monitoring and rollback evidence. Heavy approved pages remain in the release_candidate directory.'+chr(10))
zp=U/'notebook57_v8_8f_upload_bundle.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(U.iterdir()):
        if p.is_file() and p!=zp:z.write(p,p.name)
require(zp.exists() and zp.stat().st_size>0,'Upload ZIP failed.')
print(chr(10).join(summary));print('Upload:',zp)
if first_failed: raise RuntimeError(f'Notebook 57 failed closed at {first_failed}. Evidence was written to {OUT}')


## Outputs to inspect first

1. `executive_summary.txt`
2. `notebook57_verdict.json`
3. `release_candidate/release_manifest.json`
4. `release_candidate/notebook09_disposition.json`
5. `release_candidate/rollback_plan.json`
6. `release_candidate/monitoring_contract.json`
7. `release_candidate/release_page_manifest.csv`
8. `release_candidate/notebook09_exception_pages.csv`
9. `activation_record.json`
10. `upload/notebook57_v8_8f_upload_bundle.zip`

### Recommended execution

Run once with `AUTHORISE_RELEASE=False`. Inspect the candidate manifest, page scope, exception list, Notebook 09 disposition and rollback record. Only a deliberate second run with `AUTHORISE_RELEASE=True`, a named approver and an approval reference may update the versioned release registry and active-release pointer.
